# Análisis de Sincronización: Hilbert + Kuramoto + Joint Recurrence Plots (T11/T12) - Output

Resultados del análisis de sincronización entre FormacionMWEntT11TempPV y FormacionMWSalT12TempPV.

Las señales T11 y T12 ya tienen un retardo natural (físico) entre ellas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Load saved results
results_df = pd.read_parquet('../data/kuramoto_jrp_lag_sweep_T11_T12.parquet')
print(f'Loaded {len(results_df)} lag points')
print(results_df.head())

In [ ]:
# Summary statistics
best_kuramoto = int(results_df.loc[results_df['kuramoto_score'].idxmax(), 'lag'])
best_kuramoto_sustained = results_df.loc[results_df['kuramoto_score'].idxmax(), 'max_sustained_sec']
best_kuramoto_frac = results_df.loc[results_df['kuramoto_score'].idxmax(), 'frac_above_07']
best_kuramoto_r = results_df.loc[results_df['kuramoto_score'].idxmax(), 'r_mean']

jrp_valid = results_df[(results_df['lag'] != -results_df['lag'].max()) & (results_df['lag'] != results_df['lag'].max()) & (results_df['jrp_RR'] > 0)].copy()
best_jrp = int(jrp_valid.loc[jrp_valid['jrp_score'].idxmax(), 'lag']) if len(jrp_valid) > 0 else 'N/A'
best_combined = int(results_df.loc[results_df['combined_score'].idxmax(), 'lag'])

print('=== FINAL SUMMARY ===')
print(f'Best lag (Kuramoto - frac_above_07 × r_mean):     {best_kuramoto}s ({best_kuramoto/60:.1f} min)  [max_sustained={best_kuramoto_sustained:.0f}s, frac_above_07={best_kuramoto_frac:.3f}, r_mean={best_kuramoto_r:.3f}]')
print(f'Best lag (JRP - DET/LAM/RR, excl. extreme lags): {best_jrp}s ({best_jrp/60:.1f}s)')
print(f'Best lag (Combined):                               {best_combined}s ({best_combined/60:.1f} min)')

In [ ]:
# Plot Kuramoto metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(results_df['lag'], results_df['r_mean'], 'b-', linewidth=1)
axes[0, 0].set_xlabel('Lag (seconds)')
axes[0, 0].set_ylabel('Mean Kuramoto r')
axes[0, 0].set_title('Mean Kuramoto Order Parameter vs Lag')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axvline(x=0, color='k', linestyle=':', alpha=0.5)
axes[0, 0].axvline(x=best_kuramoto, color='g', linestyle='--', alpha=0.7, label=f'Best Kuramoto ({best_kuramoto}s)')
axes[0, 0].legend()

axes[0, 1].plot(results_df['lag'], results_df['frac_above_07'], 'r-', linewidth=1)
axes[0, 1].set_xlabel('Lag (seconds)')
axes[0, 1].set_ylabel('Fraction r > 0.7')
axes[0, 1].set_title('Fraction of Time with Strong Phase Sync')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(x=0, color='k', linestyle=':', alpha=0.5)
axes[0, 1].axvline(x=best_kuramoto, color='g', linestyle='--', alpha=0.7, label=f'Best Kuramoto ({best_kuramoto}s)')
axes[0, 1].legend()

axes[1, 0].plot(results_df['lag'], results_df['max_sustained_sec'], 'g-', linewidth=1)
axes[1, 0].set_xlabel('Lag (seconds)')
axes[1, 0].set_ylabel('Max sustained (seconds)')
axes[1, 0].set_title('Longest Continuous Sync Segment')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axvline(x=0, color='k', linestyle=':', alpha=0.5)
axes[1, 0].axvline(x=best_kuramoto, color='g', linestyle='--', alpha=0.7, label=f'Best Kuramoto ({best_kuramoto}s)')
axes[1, 0].legend()

axes[1, 1].plot(results_df['lag'], results_df['r_std'], 'm-', linewidth=1)
axes[1, 1].set_xlabel('Lag (seconds)')
axes[1, 1].set_ylabel('Std Kuramoto r')
axes[1, 1].set_title('Std of Kuramoto Order Parameter')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axvline(x=0, color='k', linestyle=':', alpha=0.5)
axes[1, 1].axvline(x=best_kuramoto, color='g', linestyle='--', alpha=0.7, label=f'Best Kuramoto ({best_kuramoto}s)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot JRP metrics
jrp_df = results_df[results_df['jrp_RR'] > 0].copy()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(jrp_df['lag'], jrp_df['jrp_RR'], 'b-o', linewidth=1, markersize=3)
axes[0, 0].set_xlabel('Lag (seconds)')
axes[0, 0].set_ylabel('Joint Recurrence Rate')
axes[0, 0].set_title('JRP: Recurrence Rate vs Lag')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axvline(x=0, color='k', linestyle=':', alpha=0.5)
if best_jrp != 'N/A':
    axes[0, 0].axvline(x=best_jrp, color='g', linestyle='--', alpha=0.7, label=f'Best JRP ({best_jrp}s)')
axes[0, 0].legend()

axes[0, 1].plot(jrp_df['lag'], jrp_df['jrp_DET'], 'r-o', linewidth=1, markersize=3)
axes[0, 1].set_xlabel('Lag (seconds)')
axes[0, 1].set_ylabel('Determinism (DET)')
axes[0, 1].set_title('JRP: Determinism vs Lag')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].axvline(x=0, color='k', linestyle=':', alpha=0.5)
if best_jrp != 'N/A':
    axes[0, 1].axvline(x=best_jrp, color='g', linestyle='--', alpha=0.7, label=f'Best JRP ({best_jrp}s)')
axes[0, 1].legend()

axes[1, 0].plot(jrp_df['lag'], jrp_df['jrp_LAM'], 'g-o', linewidth=1, markersize=3)
axes[1, 0].set_xlabel('Lag (seconds)')
axes[1, 0].set_ylabel('Laminarity (LAM)')
axes[1, 0].set_title('JRP: Laminarity vs Lag')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].axvline(x=0, color='k', linestyle=':', alpha=0.5)
if best_jrp != 'N/A':
    axes[1, 0].axvline(x=best_jrp, color='g', linestyle='--', alpha=0.7, label=f'Best JRP ({best_jrp}s)')
axes[1, 0].legend()

axes[1, 1].plot(jrp_df['lag'], jrp_df['jrp_max_diag'], 'm-o', linewidth=1, markersize=3)
axes[1, 1].set_xlabel('Lag (seconds)')
axes[1, 1].set_ylabel('Max Diagonal Line')
axes[1, 1].set_title('JRP: Max Diagonal Length vs Lag')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axvline(x=0, color='k', linestyle=':', alpha=0.5)
if best_jrp != 'N/A':
    axes[1, 1].axvline(x=best_jrp, color='g', linestyle='--', alpha=0.7, label=f'Best JRP ({best_jrp}s)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Combined scores plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(results_df['lag'], results_df['kuramoto_score'], 'b-', linewidth=1)
axes[0].set_xlabel('Lag (seconds)')
axes[0].set_ylabel('Kuramoto Score (normalized frac_above_07 × r_mean)')
axes[0].set_title('Kuramoto Score vs Lag')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=0, color='k', linestyle=':', alpha=0.5)
axes[0].axvline(x=best_kuramoto, color='g', linestyle='--', alpha=0.7, label=f'Best ({best_kuramoto}s)')
axes[0].legend()

axes[1].plot(jrp_df['lag'], jrp_df['jrp_score'], 'r-o', linewidth=1, markersize=3)
axes[1].set_xlabel('Lag (seconds)')
axes[1].set_ylabel('JRP Score (normalized DET×LAM×RR)')
axes[1].set_title('JRP Score vs Lag')
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=0, color='k', linestyle=':', alpha=0.5)
if best_jrp != 'N/A':
    axes[1].axvline(x=best_jrp, color='g', linestyle='--', alpha=0.7, label=f'Best ({best_jrp}s)')
axes[1].legend()

axes[2].plot(results_df['lag'], results_df['combined_score'], 'g-', linewidth=1)
axes[2].set_xlabel('Lag (seconds)')
axes[2].set_ylabel('Combined Score')
axes[2].set_title('Combined Score (Kuramoto + JRP) / 2')
axes[2].grid(True, alpha=0.3)
axes[2].axvline(x=0, color='k', linestyle=':', alpha=0.5)
axes[2].axvline(x=best_combined, color='g', linestyle='--', alpha=0.7, label=f'Best ({best_combined}s)')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 results
print('=== Top 10 by Kuramoto Score (frac_above_07 × r_mean) ===')
print(results_df.nlargest(10, 'kuramoto_score')[['lag', 'r_mean', 'frac_above_07', 'max_sustained_sec', 'kuramoto_score']].to_string(index=False))

jrp_valid = results_df[(results_df['lag'] != -results_df['lag'].max()) & (results_df['lag'] != results_df['lag'].max()) & (results_df['jrp_RR'] > 0)].copy()
print('\n=== Top 10 by JRP Score (excl. extreme lags) ===')
print(jrp_valid.nlargest(10, 'jrp_score')[['lag', 'jrp_RR', 'jrp_DET', 'jrp_LAM', 'jrp_max_diag', 'jrp_score']].to_string(index=False))

print('\n=== Top 10 by Combined Score ===')
print(results_df.nlargest(10, 'combined_score')[['lag', 'r_mean', 'frac_above_07', 'max_sustained_sec', 'jrp_RR', 'jrp_DET', 'combined_score']].to_string(index=False))